# import dependencies

In [17]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import spacy
import pandas as pd

# Load and prepare Dataset

In [18]:
urdu_data = pd.read_pickle('data.pkl')
urdu_data

,Text,Type,Short Summary
0,دُنیا کی تاریخ زمین اور سمندروں پر ہونے والے ت...,History,سمندر کی گہرائیوں میں پائی جانے والی یہ معدنیا...
1,عام طور پر 5-6 ملی میٹر کی پتھری پیشاب کے ذریع...,Health,تاہم اگر مثانے میں پتھری ہو یا سائز میں چھوٹی ...
2,کوڑے کیسے شروع ہوئے؟\nشروع میں دالوں کو پیس کر...,History,کئی جگہوں پر اس کا ذکر بھی آیا ہے کہ پکوڑے یا ...
3,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔یہ ابت...,Nature,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔
4,دل کو تقویت دیتا ہے۔قے اور بخار سے نجات دلاتا ...,General,دل کو تقویت دیتا ہے۔
...,...,...,...
995,کیوی پھل میں الکلی کی خاصیتیں موجود ہیں جو تیز...,Nature,کیوی پھل میں الکلی کی خاصیتیں موجود ہیں جو تیز...
996,السلام علیکم، عید مبارک ہو۔ و علیکم السلام، آ...,General,السلام علیکم، عید مبارک ہو۔
997,واشنگٹن (اردو پوائنٹ تازہ ترین اخبار ۔ 13 فرور...,Food,عاصم خان نے سیمی فائنل میں اپنے ہم وطن عبدالما...
998,پس جب کسی مسئلے سے ایسے بد نتیجے نکلیں تو مجھ ...,General,پس جب کسی مسئلے سے ایسے بد نتیجے نکلیں تو مجھ ...


In [19]:
test_data = urdu_data[900:]
urdu_data = urdu_data[0:900]

In [20]:
# Use spaCy for sentence tokenization
nlp = spacy.blank('ur')
nlp.add_pipe('sentencizer')

In [21]:
def tokenize_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]
    
# Tokenize all documents into sentences
urdu_data['sentences'] = urdu_data['Text'].apply(tokenize_sentences)

/tmp/ipykernel_2638771/547752441.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  urdu_data['sentences'] = urdu_data['Text'].apply(tokenize_sentences)


In [22]:
# Flatten all sentences into a single list to build vocabulary and sequence data
all_sentences = [sent for sublist in urdu_data['sentences'] for sent in sublist]

In [23]:
# Initialize and fit the tokenizer
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(all_sentences)

In [24]:
sentence_sequences = [tokenizer.texts_to_sequences(sents) for sents in urdu_data['sentences']]

In [25]:
max_length = max(len(seq) for sublist in sentence_sequences for seq in sublist)
padded_sequences = [pad_sequences(sublist, maxlen=max_length, padding='post') for sublist in sentence_sequences]

In [26]:
# Create a flat list of sequences and a corresponding label list
X = np.concatenate(padded_sequences)
labels = []

# Create labels based on inclusion in the short summary
for idx, row in urdu_data.iterrows():
    summary_sentences = tokenize_sentences(row['Short Summary'])
    document_sentences = row['sentences']
    labels += [1 if any(sen in doc for sen in summary_sentences) else 0 for doc in document_sentences]

y = np.array(labels)

In [27]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
max_length

192

# Build and train model

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional, Dropout

# Model configuration
embedding_dim = 100

# Building the model
model = Sequential([
    Embedding(input_dim=5000, output_dim=embedding_dim, input_length=max_length),
    Bidirectional(LSTM(64, return_sequences=False)),  # Return sequences to get output for each input
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 192, 100)          500000    
                                                                 
 bidirectional (Bidirectiona  (None, 128)              84480     
 l)                                                              
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 584,609
Trainable params: 584,609
Non-trainable params: 0
_________________________________________________________________


2024-05-13 10:19:42.737648: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2024-05-13 10:19:42.739088: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2024-05-13 10:19:42.739918: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

In [36]:
# Training the model
model.fit(X_train, y_train, epochs=3, batch_size= 5, validation_split=0.2)

Epoch 1/3


2024-05-13 10:19:47.243261: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2024-05-13 10:19:47.244675: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2024-05-13 10:19:47.245670: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

540/540 [==============================] - ETA: 0s - loss: 0.2748 - accuracy: 0.8921

2024-05-13 10:20:14.591913: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2024-05-13 10:20:14.593354: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2024-05-13 10:20:14.594225: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

540/540 [==============================] - 28s 48ms/step - loss: 0.2748 - accuracy: 0.8921 - val_loss: 0.0535 - val_accuracy: 0.9793
Epoch 2/3
540/540 [==============================] - 7s 13ms/step - loss: 0.0319 - accuracy: 0.9892 - val_loss: 0.0245 - val_accuracy: 0.9881
Epoch 3/3
540/540 [==============================] - 7s 13ms/step - loss: 0.0179 - accuracy: 0.9911 - val_loss: 0.0201 - val_accuracy: 0.9941


In [37]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")
model.save('LSTM_Summarizer.h5')

2024-05-13 10:20:41.261191: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2024-05-13 10:20:41.262394: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2024-05-13 10:20:41.263195: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

27/27 [==============================] - 1s 7ms/step - loss: 0.0234 - accuracy: 0.9917
Test Loss: 0.023383811116218567
Test Accuracy: 0.991696298122406


# Test model

In [39]:
import tensorflow as tf 
model = tf.keras.models.load_model('LSTM_Summarizer.h5')

2024-05-13 10:20:52.790323: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2024-05-13 10:20:52.791503: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2024-05-13 10:20:52.792305: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

In [40]:
urdu_data = pd.read_pickle('data.pkl')
test_data = urdu_data[850:999]

def generate_summary(text, model, tokenizer, max_length):
    # Tokenize text
    nlp = spacy.blank('ur')  # Load the appropriate language model
    nlp.add_pipe('sentencizer')
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    # Convert sentences to sequences and pad them
    sequences = tokenizer.texts_to_sequences(sentences)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

    # Get predictions from the model
    predictions = model.predict(padded_sequences)
    predicted_sentences = [s for s, p in zip(sentences, predictions) if p > 0.5]

    # Combine the selected sentences to form a summary
    return ' '.join(predicted_sentences)

# Example usage
new_text = urdu_data['Text'][510]
summary = generate_summary(new_text, model, tokenizer,  max_length)
print('Text',new_text)
print("Predicted Summary:", summary)
print('Orignal Summary',urdu_data['Short Summary'][510])


2024-05-13 10:20:54.324866: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2024-05-13 10:20:54.325869: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2024-05-13 10:20:54.326957: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

1/1 [==============================] - 0s 437ms/step
Text ایک عام تاثر یہ ہے کہ سمارٹ فونز کی وجہ سے لوگ اپنی سماجی سرگرمیوں سے دور ہوتے جا رہے ہیں۔ لیکن اگر آپ اس پہلو پر ذرا غور کریں تو آپ کو اندازہ ہوگا کہ سمارٹ فونز سماجی دنیا سے آپ کا رابطہ اور ہموار کر رہے ہیں نا کہ کم کر رہے ہیں۔اور اب تو یہ بات تحقیق سے بھی ثابت ہو چکی ہے ۔ آپ کو ایسا لگتا ہوگا کہ سمارٹ فونز لوگوں کو ان کے ارد گرد ہونے والی واقعات سے بے خبر کر دیتے ہیں مگر حقیقیت اس کے برعکس ہے۔

2008 میں کیتھ ہیپٹن نے اس بارے میں تحقیق کرنے کے لیے نیو یارک کے برینٹ پارک او ر آرٹ کے میٹروپولیٹن میوزیم میں کچھ کیمرے لگوائے اور ان جگہوں کا 1970کی فوٹیج سے موازنہ کیا۔ان جگہوں پر لوگوں کا مشاہدہ کرنے کے بعد وہ اس نتیجے پر پہنچی کہ سمارٹ فونز گروپس کی شکل میں استعمال کیے جاتے ہیں۔
Predicted Summary: ایک عام تاثر یہ ہے کہ سمارٹ فونز کی وجہ سے لوگ اپنی سماجی سرگرمیوں سے دور ہوتے جا رہے ہیں۔
Orignal Summary ایک عام تاثر یہ ہے کہ سمارٹ فونز کی وجہ سے لوگ اپنی سماجی سرگرمیوں سے دور ہوتے جا رہے ہیں۔


In [56]:
orig = []
pred = []
for i in range(900,920):
    summary = generate_summary(test_data['Text'][i], model, tokenizer,  max_length)
    pred.append(summary)
    orig.append(test_data['Short Summary'][i])

1/1 [==============================] - 0s 17ms/step


In [60]:
from rouge_scores import calculate_rouge_scores
rouge_scores = calculate_rouge_scores(pred, orig)
print("ROUGE Scores:", rouge_scores)

ROUGE Scores: {'rouge1': 0.5, 'rouge2': 0.0, 'rougeL': 0.5}
